In [13]:
x = torch.tensor([[2,2,2],[1,1,1],[8,8,8]], dtype=torch.float32)
torch.mean(x, 0)

tensor([3.6667, 3.6667, 3.6667])

In [ ]:
# 这个例子展示了如何使用矩阵乘法来实现“加权聚合”，
# 它将一个序列中的元素进行加权平均，其中权重由一个下三角矩阵确定。
torch.manual_seed(42)
a = torch.tril(torch.ones(3, 3))
a = a / torch.sum(a, 1, keepdim=True)
b = torch.randint(0,10,(3,2)).float()
c = a @ b
print('a=')
print(a)
print('--')
print('b=')
print(b)
print('--')
print('c=')
print(c)

a=
tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])
--
b=
tensor([[2., 7.],
        [6., 4.],
        [6., 5.]])
--
c=
tensor([[2.0000, 7.0000],
        [4.0000, 5.5000],
        [4.6667, 5.3333]])


In [17]:
# first version
# 这段代码在做“因果前缀均值”（running mean）：对每个样本 b、每个时间步 t，把从第 0 步到 t 步的特征向量按均值聚合，得到 xbow[b, t]。
# 具体流程：    
# 设定随机种子，生成形状为 (B, T, C) 的随机张量 x。
# 初始化输出 xbow 为全 0，同形状 (B, T, C)。
# 双重循环：对每个 b、每个 t，取前缀 x[b, :t+1]（形状 (t+1, C)），在时间维上求均值，得到 (C,)；将其写入 xbow[b, t]。
# 打印语句只是用来核对前缀张量与均值的形状/数值。
# 作用与直觉：这是一个“只看过去”的因果聚合示例，相当于“自注意力里用均匀权重”的特例，用来演示如何将过去的上下文压缩成当前时刻的表示。

import torch
import torch.nn as nn
from torch.nn import functional as F

torch.manual_seed(1557)
B,T,C = 4,8,2 # batch, time, channels
x = torch.randn(B,T,C)
x.shape

# We want x[b,t] = mean_{i<=t} x[b,i]
xbow = torch.zeros((B,T,C))
for b in range(B):
    for t in range(T):
        xprev = x[b,:t+1] # (t,C)
        print(xprev.shape)
        print("torch mean is", torch.mean(xprev, 0), "shape is", torch.mean(xprev, 0).shape)
        xbow[b,t] = torch.mean(xprev, 0)

torch.Size([1, 2])
torch mean is tensor([-1.0557,  0.4534]) shape is torch.Size([2])
torch.Size([2, 2])
torch mean is tensor([-1.2324,  0.5479]) shape is torch.Size([2])
torch.Size([3, 2])
torch mean is tensor([-0.4785,  0.5867]) shape is torch.Size([2])
torch.Size([4, 2])
torch mean is tensor([0.0700, 0.2829]) shape is torch.Size([2])
torch.Size([5, 2])
torch mean is tensor([0.1297, 0.4357]) shape is torch.Size([2])
torch.Size([6, 2])
torch mean is tensor([-0.0071,  0.3669]) shape is torch.Size([2])
torch.Size([7, 2])
torch mean is tensor([-0.0384,  0.2635]) shape is torch.Size([2])
torch.Size([8, 2])
torch mean is tensor([0.2142, 0.4896]) shape is torch.Size([2])
torch.Size([1, 2])
torch mean is tensor([0.3955, 2.2030]) shape is torch.Size([2])
torch.Size([2, 2])
torch mean is tensor([-0.4409,  1.1066]) shape is torch.Size([2])
torch.Size([3, 2])
torch mean is tensor([-0.2820,  0.4781]) shape is torch.Size([2])
torch.Size([4, 2])
torch mean is tensor([-0.1663,  0.2487]) shape is torc

In [18]:
# version 2: using matrix multiply for a weighted aggregation
wei = torch.tril(torch.ones(T, T))
wei = wei / wei.sum(1, keepdim=True)
xbow2 = wei @ x # (B, T, T) @ (B, T, C) ----> (B, T, C)
torch.allclose(xbow, xbow2)

True

In [22]:
# version 3: use Softmax
tril = torch.tril(torch.ones(T, T))
wei = torch.zeros((T,T))
wei = wei.masked_fill(tril == 0, float('-inf'))
print("wei is", wei)
wei = F.softmax(wei, dim=-1)
print("after softmax", wei)
xbow3 = wei @ x
torch.allclose(xbow, xbow3)

wei is tensor([[0., -inf, -inf, -inf, -inf, -inf, -inf, -inf],
        [0., 0., -inf, -inf, -inf, -inf, -inf, -inf],
        [0., 0., 0., -inf, -inf, -inf, -inf, -inf],
        [0., 0., 0., 0., -inf, -inf, -inf, -inf],
        [0., 0., 0., 0., 0., -inf, -inf, -inf],
        [0., 0., 0., 0., 0., 0., -inf, -inf],
        [0., 0., 0., 0., 0., 0., 0., -inf],
        [0., 0., 0., 0., 0., 0., 0., 0.]])
after softmax tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.0000, 0.0000, 0.0000],
        [0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.0000, 0.0000],
        [0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.0000],
        [0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1

True